In [1]:
# Core analysis tools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import sys
import os

# Add repo root to path
sys.path.insert(
    0,
    os.path.abspath("..")
)

# Custom analysis modules
from src.io import load_samples
from src import preprocessing
from src import dim_reduction as dr
from src import expression
from src import markers
from src import annotation
from src import gep
from src import plotting

## Get Processed Counts from CellRanger output and Analyze Expression

Select a parent directory, then loop across the subdirectories to extract the single-cell matrices and compare using Scanpy. Subdirectory names should correspond to condition and extract two files: the AnnData file (`.h5`/`.h5ad`) and the counts file (`.tsv.gz`). Then, concatenate across samples.

Here, manually input the path and sample names for each run.

In [2]:
parent_dir = "/Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260812"
samples = ["GD4", "GD4C", "GD18", "GD18C"]

adata_all = load_samples(parent_dir, samples)

adata_all.obs["condition"] = (
    adata_all.obs["sample"]
    .map({
        "GD4": "GD4-preg",
        "GD4C": "GD4-control",
        "GD18": "GD18-preg",
        "GD18C": "GD18-control"
    })
)

Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260812/GD4/filtered_feature_bc_matrix.h5


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Loaded 22,351 cells × 55,420 genes
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260812/GD4C/filtered_feature_bc_matrix.h5


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Loaded 21,841 cells × 55,420 genes
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260812/GD18/filtered_feature_bc_matrix.h5


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Loaded 18,838 cells × 55,420 genes
Loading /Users/norawolcott/Documents/Datta Lab/data/scRNA-seq/260812/GD18C/filtered_feature_bc_matrix.h5


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


  Loaded 23,477 cells × 55,420 genes


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Find the most differentially expressed genes across cells

In [3]:
# Store raw counts
adata_all = preprocessing.store_raw_counts(adata_all)

# Normalize (and calculate total counts if not already counted)
adata_all = preprocessing.normalize_data(adata_all)

# Select HVGs
adata_all = preprocessing.identify_hvgs(
    adata_all,
    n_top_genes=3000
)

top10_hvg = (
    adata_all.var[adata_all.var["highly_variable"]]
    .sort_values("dispersions_norm", ascending=False)
    .head(10)
)

print('Top 10 most differentially expressed genes across samples:')
print(top10_hvg.index.tolist())

OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Top 10 most differentially expressed genes across samples:
['Bpifa1', 'Wfdc18', 'Scgb1c1', 'Hbb-bt', 'Apod', 'S100a9', 'S100a8', 'Hba-a1', 'Hba-a2', 'Rgs5']


In [4]:
X = adata_all.layers["counts"].data

print("Integer counts:", np.array_equal(X, np.round(X)))
print("Min:", X.min())
print("Max:", X.max())

Integer counts: True
Min: 1.0
Max: 22921.0


## Dimensionality Reduction
Perform PCA, KNN, UMAP, and Leiden analysis.

In [5]:
# PCA
adata_all, variance = dr.run_pca(adata_all)

print("Variance explained:")
for i, v in enumerate(variance[:10]):
    print(f"PC{i+1}: {v*100:.2f}%")

print(
    f"\nTotal variance explained by first 10 PCs: "
    f"{variance[:10].sum()*100:.2f}%"
)

# Dimensionality reduction
adata_all = dr.run_umap(
    adata_all,
    n_pcs=35
)

adata_all = dr.leiden_clustering(
    adata_all,
    resolution=0.5
)

# Plot embeddings
plotting.plot_umap_metadata(adata_all)
plotting.plot_umap_clusters(adata_all)


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:12: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.tl.pca(


Variance explained:
PC1: 22.25%
PC2: 10.45%
PC3: 4.77%
PC4: 4.40%
PC5: 3.63%
PC6: 2.71%
PC7: 2.49%
PC8: 1.96%
PC9: 1.75%
PC10: 1.46%

Total variance explained by first 10 PCs: 55.86%


/opt/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:51: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(


Analyze expression of HVGs and top marker genes.

In [6]:
# Find marker genes
adata_all = expression.rank_cluster_markers(
    adata_all,
    groupby="leiden"
)

# Get top markers
top_genes = expression.get_top_marker_genes(
    adata_all
)

print("Top DE gene for each Leiden group:")
for group, genes in top_genes.items():
    print(f"  {group}: {genes[0]}")

Top DE gene for each Leiden group:
  0: S100a5
  1: Acsm4
  2: Gramd1c
  3: mt-Rnr2
  4: Gap43
  5: Ptma
  6: Stmn2
  7: Cd36
  8: Coch
  9: Cbr2
  10: Cd36
  11: B2m
  12: Hes6
  13: Adgrg6
  14: Sparc
  15: Stap1
  16: Hbb-bs
  17: Cyp2a5


## Identify mature OSNs

Score data according to marker genes (specified in markers.py).

In [7]:
marker_sets = {
    "mature_OSN_score": markers.mature_osn_markers,
    "immature_OSN_score": markers.immature_osn_markers,
    "non_neuronal_score": markers.non_neuronal_markers,
    "osn_score": markers.osn_markers,
    "epithelial_score": markers.epithelial_markers
}

adata_all = annotation.score_gene_sets(
    adata_all,
    marker_sets
)

In [8]:
print(adata_all.obs.columns.tolist())

['sample', 'condition', 'total_counts', 'n_genes_by_counts', 'pct_counts_mito', 'leiden', 'mature_OSN_score', 'immature_OSN_score', 'non_neuronal_score', 'osn_score', 'epithelial_score']


Plot UMAPs of marker genes

In [9]:
# Plot OSN scoring metrics
plotting.plot_score_umap(
    adata_all
)

plotting.plot_marker_umap(
    adata_all,
    genes=markers.osn_markers,
    cmap="viridis_r",
    filename="osn_marker_umaps.png"
)

plotting.plot_marker_umap(
    adata_all,
    genes=markers.epithelial_markers,
    cmap="viridis_r",
    filename="epithelial_marker_umaps.png"
)

Identify mature OSN clusters

In [10]:
likely_mature_osns, cluster_scores = annotation.identify_mature_osn_clusters(
    adata_all
)

Cluster score summary:
        mature_OSN_score  immature_OSN_score  non_neuronal_score  \
leiden                                                             
0               1.747800           -0.438342           -0.077967   
1               1.688387           -0.345866           -0.072653   
2               1.694044           -0.215325           -0.077773   
3               1.096150           -0.039740            0.018117   
4              -0.183339            0.963586           -0.090133   
5              -0.565014            0.978529           -0.077145   
6               1.205114            0.325085           -0.093145   
7               1.553526           -0.414834           -0.066711   
8              -0.356340           -0.324210            0.704971   
9              -0.228327           -0.310688            1.574364   
10              1.555091           -0.391236           -0.071115   
11             -0.084472           -0.279830            0.154900   
12             -0.551155 

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:30: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[


Check OR expression

In [11]:
adata_all = plotting.plot_or_gene_counts(
    adata_all
)

In [12]:
print(adata_all.obs.columns.tolist())

['sample', 'condition', 'total_counts', 'n_genes_by_counts', 'pct_counts_mito', 'leiden', 'mature_OSN_score', 'immature_OSN_score', 'non_neuronal_score', 'osn_score', 'epithelial_score', 'n_ORs']


Identify OSN-OSN doublets

In [13]:
candidate_osn_osn_doublets, cluster_doublet_screen = annotation.screen_osn_doublets(
    adata_all
)

print("Candidate OSN-OSN doublet clusters:")
print(candidate_osn_osn_doublets)

display(
    cluster_doublet_screen.sort_values(
        "total_counts",
        ascending=False
    )
)

Candidate OSN-OSN doublet clusters:
['12', '15']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:70: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[
/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:82: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)["n_ORs"]


,total_counts,n_genes_by_counts,n_ORs,pct_cells_multiple_ORs
leiden,,,,
12,20560.949219,4886.062315,2.303660,0.606330
15,19844.853516,5334.592437,3.386555,0.886555
6,18186.265625,4939.014930,4.021972,0.860282
4,13820.100586,4416.089787,3.353067,0.807011
5,12513.253906,3802.091410,6.288653,0.727041
2,11910.597656,3820.538256,2.568861,0.772189
0,11815.918945,3761.326816,2.719751,0.811315
7,11389.291992,3716.838902,2.770709,0.820595
10,11374.718750,3645.367133,2.692308,0.777098


Identify OSN + non-neuronal doublets

In [14]:
candidate_osn_nonneuronal_doublets, cluster_mixed_screen = (
    annotation.screen_osn_nonneuronal_doublets(
        adata_all
    )
)

print("Candidate OSN + non-neuronal doublet clusters:")
print(candidate_osn_nonneuronal_doublets)

display(
    cluster_mixed_screen.sort_values(
        "mature_OSN_score",
        ascending=False
    )
)

Candidate OSN + non-neuronal doublet clusters:
['3', '15', '16']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:111: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby(cluster_key)[


,mature_OSN_score,non_neuronal_score,immature_OSN_score,total_counts,pct_counts_mito
leiden,,,,,
0,1.747800,-0.077967,-0.438342,11815.918945,4.814677
2,1.694044,-0.077773,-0.215325,11910.597656,4.287649
1,1.688387,-0.072653,-0.345866,10939.630859,4.374046
10,1.555091,-0.071115,-0.391236,11374.718750,3.558187
7,1.553526,-0.066711,-0.414834,11389.291992,4.177805
6,1.205114,-0.093145,0.325085,18186.265625,2.658037
15,1.174081,0.455010,-0.294934,19844.853516,6.925033
3,1.096150,0.018117,-0.039740,1568.143799,26.457577
16,0.444883,-0.010897,-0.036872,4710.467773,0.938707


Remove OSN doublet clusters from mature OSN candidates

In [15]:
# Combine all candidate doublet clusters
all_candidate_doublets = (
    candidate_osn_osn_doublets +
    candidate_osn_nonneuronal_doublets
)

# Remove doublet clusters from mature OSN candidates
likely_mature_osns_filtered = annotation.filter_doublet_clusters(
    likely_mature_osns,
    all_candidate_doublets
)

print("Original likely mature OSN clusters:")
print(likely_mature_osns)

print("\nRemoving candidate doublet clusters:")
print(all_candidate_doublets)

print("\nFinal mature OSN clusters:")
print(likely_mature_osns_filtered)

Original likely mature OSN clusters:
['0', '1', '7', '10']

Removing candidate doublet clusters:
['12', '15', '3', '15', '16']

Final mature OSN clusters:
['0', '1', '7', '10']


Recalculate HVGs just using mature OSN clusters

In [16]:
adata_mature_osn = annotation.subset_clusters(
    adata_all,
    likely_mature_osns_filtered
)

print(f"Starting mature OSN cells: {adata_mature_osn.n_obs}")

Starting mature OSN cells: 41421


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [17]:
raw_counts = adata_mature_osn.layers["counts"]

X = raw_counts.data
print(np.array_equal(X, np.round(X)))

True


Identify mitochondrial genes

In [18]:
adata_mature_osn = preprocessing.filter_cells_by_qc(
    adata_mature_osn,
    min_counts=1000,
    max_pct_mt=10
)

print(f"Cells after QC filtering: {adata_mature_osn.n_obs}")

Cells after QC filtering: 41413


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


Re-normalize and identify HVGs again

In [19]:
adata_mature_osn = preprocessing.normalize_and_select_hvgs(
    adata_mature_osn,
    n_top_genes=3000
)

In [20]:
raw_counts = adata_mature_osn.layers["counts"]

X = raw_counts.data
print(np.array_equal(X, np.round(X)))

True


Redo PCA with only 20 PCs

In [21]:
# Dimensional reduction of mature OSNs
adata_mature_osn, variance_mature_osn = dr.run_pca(
    adata_mature_osn,
    n_comps=20
)

adata_mature_osn = dr.run_umap(
    adata_mature_osn,
    n_pcs=20
)

adata_mature_osn = dr.leiden_clustering(
    adata_mature_osn,
    resolution=1.0
)

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:12: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.tl.pca(
/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:51: FutureWarning: The `igraph` implementation of leiden clustering is *orders of magnitude faster*. Set the flavor argument to (and install if needed) 'igraph' to use it.
In the future, the default backend for leiden will be igraph instead of leidenalg. To achieve the future defaults please pass: `flavor='igraph'` and `n_iterations=2`. `directed` must also be `False` to work with igraph’s implementation.
  sc.tl.leiden(


Annotate immature neurons using marker genes

In [22]:
adata_mature_osn = annotation.score_gene_sets(
    adata_mature_osn,
    {
        "immature_OSN_score": markers.immature_osn_markers
    }
)

plotting.plot_marker_umap(
    adata_mature_osn,
    genes=[
        "immature_OSN_score",
        "Gap43",
        "Sox11",
        "Calb2",
        "pct_counts_mt",
        "total_counts"
    ],
    cmap="viridis_r",
    filename="mature_osn_immature_score_umaps.png"
)

Identify immature clusters

In [23]:
clusters_to_remove, mature_cluster_scores = (
    annotation.identify_clusters_by_score(
        adata_mature_osn,
        score_key="immature_OSN_score",
        threshold=0.2
    )
)

print("\nClusters with immature OSN score > 0.2 (remove):")
print(clusters_to_remove)


Clusters with immature OSN score > 0.2 (remove):
['8']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:192: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(cluster_key)[


Remove remaining immature OSN clusters

In [24]:
# Remove immature-like OSN clusters
adata_mature_osn = annotation.remove_clusters(
    adata_mature_osn,
    clusters_to_remove
)

print(
    f"Remaining mature OSN cells: {adata_mature_osn.n_obs}"
)


# Calculate percentage of original cells retained as mature OSNs
n_original_cells = adata_all.n_obs
n_mature_osn_cells = adata_mature_osn.n_obs

percent_mature_osns = (
    n_mature_osn_cells / n_original_cells
) * 100

print(
    f"Mature OSNs represent {percent_mature_osns:.1f}% "
    f"of the original cell population "
    f"({n_mature_osn_cells}/{n_original_cells} cells)"
)

Remaining mature OSN cells: 39401
Mature OSNs represent 45.5% of the original cell population (39401/86507 cells)


/opt/miniconda3/lib/python3.13/site-packages/anndata/_core/anndata.py:1882: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


## Analyze differential expression in just mature OSNs

Plot mature OSN UMAP colored by experimental groups

In [25]:
# Plot mature OSN UMAPs
plotting.plot_umap_clusters(
    adata_mature_osn,
    filename="mature_osn_umap_leiden_clusters.png"
)

plotting.plot_umap_metadata(
    adata_mature_osn,
    colors=["sample", "condition"],
    filename="mature_osn_umap_metadata.png"
)

Get the most differentially expressed genes across conditions

In [26]:
# Find marker genes by condition
adata_mature_osn = expression.rank_cluster_markers(
    adata_mature_osn,
    groupby="condition"
)

# Get top 20 markers
top_genes = expression.get_top_marker_genes(
    adata_mature_osn,
    n_genes=20
)

print("Top DE gene for each condition:")
for group, genes in top_genes.items():
    print(f"  {group}: {genes[0]}")

# Save results
expression.save_top_genes_csv(
    top_genes,
    filename="mature_osn_top20_DE_genes_by_condition.csv"
)

Top DE gene for each condition:
  GD4-control: H1f4
  GD4-preg: Gm10076
  GD18-control: Lars2
  GD18-preg: Uba52


'../results/mature_osn_top20_DE_genes_by_condition.csv'

OPTIONAL (uncomment to use): plot differential expression (specify groups)

In [27]:
# # Use volcano-style filtering
# # Get DE results
# sc.tl.rank_genes_groups(
#     adata_mature_osn,
#     groupby="condition",
#     groups=["GD18-preg"],
#     reference="GD18-control",
#     method="wilcoxon"
# )

# de_results = sc.get.rank_genes_groups_df(
#     adata_mature_osn,
#     group="GD18-preg"
# )

# # View strongest DE genes
# de_results[
#     (de_results["pvals_adj"] < 0.05) &
#     (abs(de_results["logfoldchanges"]) > 0.25)
# ].head(50)


# # Volcano plot
# plotting.plot_volcano(
#     de_results,
#     filename="mature_osn_GD18_vs_GD18C_volcano.png",
#     n_labels=20
# )

## Analyze differential expression in Bowman's gland cells

Score Bowman's gland markers

In [28]:
# Score Bowman's gland markers
adata_all = annotation.score_gene_sets(
    adata_all,
    {
        "bowmans_gland_score": markers.bowmans_gland_markers
    }
)

# Plot Bowman's gland score
plotting.plot_umap_clusters(
    adata_all,
    cluster_key="bowmans_gland_score",
    filename="bowmans_gland_score_umap.png"
)

Identify Bowman's gland clusters
Current threshold: 3 standard deviations above the mean bowman's gland score

In [29]:
# Calculate data-driven threshold: 2 SD above the mean
score_mean = adata_all.obs["bowmans_gland_score"].mean()
score_std = adata_all.obs["bowmans_gland_score"].std()
bowmans_threshold = score_mean + 3 * score_std

print(f"Bowman's gland threshold (mean + 3 SD): {bowmans_threshold:.3f}")

bowmans_gland_clusters, bowmans_cluster_scores = (
    annotation.identify_clusters_by_score(
        adata_all,
        score_key="bowmans_gland_score",
        threshold=bowmans_threshold
    )
)

print(f"Bowman's gland clusters (score > {bowmans_threshold:.3f}):")
print(bowmans_gland_clusters)

# Plot Leiden clusters and Bowman's gland score
plotting.plot_umap_metadata(
    adata_all,
    colors=["leiden", "bowmans_gland_score"],
    filename="bowmans_gland_clusters.png"
)

# Subset Bowman's gland cells
adata_bowmans = annotation.subset_clusters(
    adata_all,
    bowmans_gland_clusters
)

print(f"Bowman's gland cells: {adata_bowmans.n_obs}")

Bowman's gland threshold (mean + 3 SD): 0.282
Bowman's gland clusters (score > 0.282):
['8', '13', '17']


/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/annotation.py:192: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  .groupby(cluster_key)[


Bowman's gland cells: 2613


In [30]:
print("Bowman's gland cells by condition:")
display(
    adata_bowmans.obs["condition"].value_counts()
)

print("\nBowman's gland cells by sample:")
display(
    adata_bowmans.obs["sample"].value_counts()
)

Bowman's gland cells by condition:


condition
GD4-preg        670
GD4-control     666
GD18-control    641
GD18-preg       636
Name: count, dtype: int64


Bowman's gland cells by sample:


sample
GD4      670
GD4C     666
GD18C    641
GD18     636
Name: count, dtype: int64

OPTIONAL (uncomment to run): See if Bowman's gland cells are differentially expressed between conditions

In [31]:
# # Find DE genes by condition
# adata_bowmans = expression.rank_cluster_markers(
#     adata_bowmans,
#     groupby="condition"
# )

# # Extract Proestrus DE results
# bowmans_condition_de = sc.get.rank_genes_groups_df(
#     adata_bowmans,
#     group="GD18-preg"
# )

# display(bowmans_condition_de.head(20))

Perform dimensionality reduction and plot

In [32]:
# Dimensionality reduction for Bowman's gland cells
adata_bowmans = dr.run_dimensionality_reduction(
    adata_bowmans,
    n_top_genes=2000,
    n_pcs=20
)

# Plot UMAP by experimental groups
plotting.plot_umap_metadata(
    adata_bowmans,
    colors=["condition", "sample"],
    filename="bowmans_gland_umap_metadata.png"
)

/Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/src/dim_reduction.py:74: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  sc.tl.pca(


## Save adata objects to results directory

In [33]:
# Set results directory
REPO_ROOT = os.path.abspath(
    os.path.join(os.getcwd(), "..")
)

RESULTS_DIR = os.path.join(REPO_ROOT, "results")

# Save processed AnnData objects
adata_all.write(
    os.path.join(RESULTS_DIR, "adata_all.h5ad")
)

adata_mature_osn.write(
    os.path.join(RESULTS_DIR, "adata_mature_osn.h5ad")
)

adata_bowmans.write(
    os.path.join(RESULTS_DIR, "adata_bowmans.h5ad")
)

print("Saved AnnData objects to:", RESULTS_DIR)

Saved AnnData objects to: /Users/norawolcott/Documents/GitHub/single-cell-RNAseq-Analysis/results
